<a href="https://colab.research.google.com/github/dylan518/Evolution_fine_tune/blob/main/evolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sacrebleu datasets deap


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requ

In [ ]:
#!/usr/bin/env python
"""
Evolutionary Optimization of Continuous Prompt Embeddings for Code Generation

This script optimizes the initial token embeddings (continuous prompt embeddings)
for the phi-1.5 language model on a subset of the HumanEval dataset using an evolutionary algorithm.
It uses HumanEval unit tests to judge correctness. In this version, the original prompt is
prepended to the generated continuation before running the tests.
"""

import os
import re  # For extracting the function name
import numpy as np
import torch
import subprocess
import tempfile
import textwrap
import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from deap import base, creator, tools
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Set model name and device
MODEL_NAME = "microsoft/phi-1_5"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class EvolutionaryPromptOptimizer:
    def __init__(self,
                 model_name: str = MODEL_NAME,
                 subset_size: int = 5,
                 seq_length: int = 10,
                 max_new_tokens: int = 256,
                 population_size: int = 100,
                 num_generations: int = 10,
                 seed: int = 42):
        """
        Initialize the evolutionary optimizer.
        """
        self.model_name = model_name
        self.subset_size = subset_size
        self.seq_length = seq_length
        self.max_new_tokens = max_new_tokens
        self.population_size = population_size
        self.num_generations = num_generations
        self.seed = seed

        np.random.seed(self.seed)
        torch.manual_seed(self.seed)

        # Initialize logs list for storing run details
        self.logs = []

        # Load tokenizer and model
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name)
        self.model.to(device)
        self.model.eval()

        # Load HumanEval dataset and select a subset
        print("Loading the HumanEval dataset...")
        dataset = load_dataset("openai_humaneval")
        indices = np.random.choice(len(dataset['test']), size=self.subset_size, replace=False)
        self.subset_problems = [dataset['test'][int(i)] for i in indices]
        print(f"Selecting a subset of {self.subset_size} HumanEval problems...")

        self.hidden_size = self.model.config.hidden_size
        self.embedding_layer = self.model.get_input_embeddings()

        # Set up DEAP evolutionary framework
        self.toolbox, self.creator = self._initialize_deap()

    def _initialize_deap(self):
        creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
        creator.create("Individual", list, fitness=creator.FitnessMin)
        toolbox = base.Toolbox()
        toolbox.register("attr_float", lambda: np.random.uniform(-1, 1))
        num_genes = self.seq_length * self.hidden_size
        toolbox.register("individual", tools.initRepeat, creator.Individual,
                         toolbox.attr_float, n=num_genes)
        toolbox.register("population", tools.initRepeat, list, toolbox.individual)
        toolbox.register("evaluate", self.evaluate_batch)
        toolbox.register("mate", tools.cxBlend, alpha=0.5)
        toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.1)
        toolbox.register("select", tools.selTournament, tournsize=3)
        return toolbox, creator

    def run_humaneval_test(self, generated_code: str, test_code: str, prompt: str) -> bool:
        """
        Combine the original prompt, the generated continuation, and the test code.
        Dedent the final code so that any leading indentation is removed.
        Then run the combined code in a temporary Python file.

        The update here is that we extract the function name from the prompt
        (assuming it starts with a "def") and append a call to run the tests.
        """
        # Try to extract the candidate function name from the prompt.
        # Assumes that the prompt contains a function definition like: "def function_name(...):"
        match = re.search(r"def\s+(\w+)\s*\(", prompt)
        if match:
            candidate_name = match.group(1)
        else:
            candidate_name = "candidate"  # Fallback name if not found

        # Append a call to the test function at the end.
        call_line = (
            "\nif __name__ == '__main__':\n"
            f"    check({candidate_name})\n"
        )
        # Combine prompt, generated code, test code, and the call to check the function.
        raw_code = prompt + "\n" + generated_code + "\n" + test_code + call_line
        combined_code = textwrap.dedent(raw_code)

        with tempfile.NamedTemporaryFile(suffix=".py", delete=False) as tmp:
            test_file = tmp.name
            tmp.write(combined_code.encode("utf-8"))
        try:
            result = subprocess.run(["python3", test_file],
                                    capture_output=True,
                                    text=True)
            passed = (result.returncode == 0)
            # Log the test run details in a readable format
            log_entry = (
                "Prompt:\n" + prompt + "\n" +
                "Generated Code:\n" + generated_code + "\n" +
                "Test Code:\n" + test_code + "\n" +
                "Call Line:\n" + call_line + "\n" +
                "Combined Code:\n" + combined_code + "\n" +
                f"Return code: {result.returncode}\n" +
                "Stdout:\n" + result.stdout + "\n" +
                "Stderr:\n" + result.stderr + "\n" +
                f"Test Passed: {passed}\n" +
                "="*40 + "\n"
            )
            self.logs.append(log_entry)
        finally:
            if os.path.exists(test_file):
                os.remove(test_file)
        return passed

    def evaluate_batch(self, individuals):
        """
        Evaluate a batch of individuals (each an embedding vector) on each problem.
        For each problem, use its prompt and test code to create the full code.
        Count the number of passed tests per individual.
        """
        batch_size = len(individuals)
        num_problems = len(self.subset_problems)
        # Convert individuals into a tensor and reshape them to (batch_size, seq_length, hidden_size)
        prompt_embeddings_batch = torch.tensor(individuals, device=device, dtype=torch.float32)
        if self.seq_length > 0:
            prompt_embeddings_batch = prompt_embeddings_batch.view(batch_size, self.seq_length, self.hidden_size)
        passed_counts = np.zeros(batch_size, dtype=float)
        mini_batch_size = 50  # adjust if needed

        for problem in self.subset_problems:
            # The prompt (which should define the function) is provided in the problem.
            problem_prompt = problem['prompt']
            test_code = problem['test']

            # Tokenize the prompt and obtain its input embeddings and input_ids
            inputs = self.tokenizer(problem_prompt, return_tensors='pt').to(device)
            input_ids = inputs['input_ids']
            attention_mask = inputs['attention_mask']
            with torch.no_grad():
                inputs_embeds = self.embedding_layer(input_ids)

            # Process individuals in mini-batches
            for start_idx in range(0, batch_size, mini_batch_size):
                end_idx = min(start_idx + mini_batch_size, batch_size)
                if self.seq_length == 0:
                    # If seq_length==0, use the base model generation using input_ids and attention_mask.
                    expanded_input_ids = input_ids.expand(end_idx - start_idx, -1)
                    expanded_attention_mask = attention_mask.expand(end_idx - start_idx, -1)
                    with torch.no_grad():
                        generated_outputs = self.model.generate(
                            input_ids=expanded_input_ids,
                            attention_mask=expanded_attention_mask,
                            max_new_tokens=self.max_new_tokens,
                            do_sample=False,
                            num_return_sequences=1,
                            eos_token_id=self.tokenizer.eos_token_id,
                            pad_token_id=self.tokenizer.eos_token_id,
                        )
                else:
                    # Process the individuals' custom embeddings.
                    mini_batch_embeddings = prompt_embeddings_batch[start_idx:end_idx]
                    inputs_embeds_expanded = inputs_embeds.expand(end_idx - start_idx, -1, -1).clone()
                    attention_mask_expanded = attention_mask.expand(end_idx - start_idx, -1)
                    # Replace the first seq_length embeddings with the individual's embeddings.
                    inputs_embeds_expanded[:, :self.seq_length, :] = mini_batch_embeddings
                    with torch.no_grad():
                        generated_outputs = self.model.generate(
                            inputs_embeds=inputs_embeds_expanded,
                            attention_mask=attention_mask_expanded,
                            max_new_tokens=self.max_new_tokens,
                            do_sample=False,
                            num_return_sequences=1,
                            eos_token_id=self.tokenizer.eos_token_id,
                            pad_token_id=self.tokenizer.eos_token_id,
                        )
                generated_codes = self.tokenizer.batch_decode(generated_outputs, skip_special_tokens=True)
                print(f"Generated codes for batch starting at index {start_idx}:")
                print(generated_codes)
                # For each generated code, run the HumanEval test using the original prompt.
                for idx, gen_code in enumerate(generated_codes, start=start_idx):
                    if self.run_humaneval_test(gen_code, test_code, problem_prompt):
                        passed_counts[idx] += 1

        # Compute fitness (1 - fraction_passed)
        fractions_passed = passed_counts / num_problems
        fitnesses = [(1 - frac,) for frac in fractions_passed]
        return fitnesses

    def run_evolution(self):
        """
        Execute the main evolutionary loop.
        """
        population = self.toolbox.population(n=self.population_size)
        print("Evaluating initial population...")
        fitnesses = self.toolbox.evaluate(population)
        for ind, fit in zip(population, fitnesses):
            ind.fitness.values = fit

        for gen in range(self.num_generations):
            print(f"-- Generation {gen} --")
            offspring = self.toolbox.select(population, len(population))
            offspring = list(map(self.toolbox.clone, offspring))

            # Apply crossover and mutation
            for i in range(0, len(offspring), 2):
                if i + 1 < len(offspring) and np.random.rand() < 0.5:
                    self.toolbox.mate(offspring[i], offspring[i+1])
                    del offspring[i].fitness.values
                    del offspring[i+1].fitness.values

            for mutant in offspring:
                if np.random.rand() < 0.2:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            population[:] = offspring
            invalid_inds = [ind for ind in population if not ind.fitness.valid]
            print(f"Evaluating {len(invalid_inds)} individuals...")
            if invalid_inds:
                batch_size = 100  # adjust based on GPU memory
                for start_idx in range(0, len(invalid_inds), batch_size):
                    end_idx = min(start_idx + batch_size, len(invalid_inds))
                    batch = invalid_inds[start_idx:end_idx]
                    fitnesses = self.toolbox.evaluate(batch)
                    for ind, fit in zip(batch, fitnesses):
                        ind.fitness.values = fit

            fits = [ind.fitness.values[0] for ind in population]
            avg_fitness = sum(fits) / len(fits)
            min_fitness = min(fits)
            max_fitness = max(fits)
            avg_fraction_passed = 1 - avg_fitness
            print(f"  Min fitness (1 - fraction_passed): {min_fitness}")
            print(f"  Max fitness (1 - fraction_passed): {max_fitness}")
            print(f"  Avg fitness (1 - fraction_passed): {avg_fitness}")
            print(f"  Avg Fraction of Passed Tests: {avg_fraction_passed}")

        print("Optimization complete.")

        # At the end of the run, write the logged details to a new file in a designated folder.
        log_folder = "run_logs"
        if not os.path.exists(log_folder):
            os.makedirs(log_folder)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        log_filename = os.path.join(log_folder, f"run_log_{timestamp}.txt")
        with open(log_filename, "w") as f:
            f.write("\n".join(self.logs))
        print(f"Run logs saved to {log_filename}")


Using device: cuda


In [ ]:
def main():
    optimizer = EvolutionaryPromptOptimizer(
        model_name="microsoft/phi-1_5",
        subset_size=10,
        seq_length=10,
        max_new_tokens=350,
        population_size=20,
        num_generations=1,
        seed=1
    )
    optimizer.run_evolution()

if __name__ == "__main__":
    main()

Loading tokenizer...
Loading model...
Loading the HumanEval dataset...
Selecting a subset of 10 HumanEval problems...
Evaluating initial population...
Generated codes for batch starting at index 0:
['    if base < 2:\n        raise ValueError("base must be greater than or equal to 2")\n    if base == 10:\n        return str(x)\n    digits = []\n    while x > 0:\n        digits.append(str(x % base))\n        x //= base\n    digits.reverse()\n    return "".join(digits)\n\n\ndef change_base_to_int(x, base):\n    """Change numerical base of input number x to base.\n    return integer representation after the conversion.\n    base numbers are less than 10.\n    >>> change_base_to_int(\'22\', 3)\n    8\n    >>> change_base_to_int(\'1000\', 2)\n    7\n    >>> change_base_to_int(\'111\', 2)\n    7\n    """\n    if base < 2:\n        raise ValueError("base must be greater than or equal to 2")\n    if base == 10:\n        return x\n    digits = []\n    while x > 0:\n        digits.append(int(x %

In [ ]:
import unittest
import textwrap

# If your EvolutionaryPromptOptimizer is defined in the same notebook,
# ensure that cell is already run. If it's in a file, you might need:
#
# from evolutionary_prompt_optimizer import EvolutionaryPromptOptimizer
#

class TestEvolutionaryPromptOptimizer(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        """
        One-time setup for the optimizer to avoid re-initializing the model repeatedly.
        Adjust parameters to keep it small/fast if desired.
        """
        cls.optimizer = EvolutionaryPromptOptimizer(
            model_name="microsoft/phi-1_5",
            subset_size=1,       # minimal subset for demonstration
            seq_length=4,        # small number of embeddings for speed
            max_new_tokens=16,   # small generation length
            population_size=2,   # we won't actually run 'run_evolution' in these tests
            num_generations=1,
            seed=42
        )

    def test_run_humaneval_test_passes(self):
        """
        Provide a prompt defining a trivial function called 'candidate',
        then a generated code snippet that completes it correctly (adding a+b).
        The test code calls check(candidate), expecting arithmetic correctness.
        """
        # Prompt: function signature + docstring, no body
        prompt = textwrap.dedent('''\
        def candidate(a, b):
            """
            Returns the sum of a and b.
            """
        ''')

        # 'generated_code' must be indented to belong inside the function
        generated_code = textwrap.indent('''\
return a + b
''', '    ')

        # The test code calls the function 'candidate'
        test_code = textwrap.dedent('''\
        def check(candidate):
            assert candidate(2, 2) == 4
            assert candidate(100, 1) == 101

        check(candidate)
        ''')

        # Now run the test harness
        result = self.optimizer.run_humaneval_test(
            generated_code=generated_code,
            test_code=test_code,
            prompt=prompt
        )
        self.assertTrue(result, "Expected the snippet to pass but got a failure.")

    def test_run_humaneval_test_fails(self):
        """
        Provide a similar prompt but with an incorrect snippet (a-b).
        The test code asserts multiplication, so it should fail.
        """
        # Prompt with docstring only
        prompt = textwrap.dedent('''\
        def candidate(a, b):
            """
            Returns the product of a and b.
            """
        ''')

        # Indent so that it's inside the function definition
        generated_code = textwrap.indent('''\
return a - b
''', '    ')

        # Test code calls 'candidate' expecting multiplication
        test_code = textwrap.dedent('''\
        def check(candidate):
            assert candidate(2, 2) == 4
            assert candidate(3, 5) == 15

        check(candidate)
        ''')

        result = self.optimizer.run_humaneval_test(
            generated_code=generated_code,
            test_code=test_code,
            prompt=prompt
        )
        self.assertFalse(result, "Expected the snippet to fail but it passed.")

if __name__ == "__main__":
    unittest.main(argv=[''], exit=False)


Loading tokenizer...
Loading model...
Loading the HumanEval dataset...


..
----------------------------------------------------------------------
Ran 2 tests in 4.430s

OK


Selecting a subset of 1 HumanEval problems...
------ DEBUG: run_humaneval_test ------
Combined code:
 def candidate(a, b):
    """
    Returns the product of a and b.
    """

    return a - b

def check(candidate):
    assert candidate(2, 2) == 4
    assert candidate(3, 5) == 15

check(candidate)

Return code: 1
Stdout: 
Stderr: Traceback (most recent call last):
  File "/tmp/tmp4uyzvu_j.py", line 12, in <module>
    check(candidate)
  File "/tmp/tmp4uyzvu_j.py", line 9, in check
    assert candidate(2, 2) == 4
           ^^^^^^^^^^^^^^^^^^^^
AssertionError

---------------------------------------
------ DEBUG: run_humaneval_test ------
Combined code:
 def candidate(a, b):
    """
    Returns the sum of a and b.
    """

    return a + b

def check(candidate):
    assert candidate(2, 2) == 4
    assert candidate(100, 1) == 101

check(candidate)

Return code: 0
Stdout: 
Stderr: 
---------------------------------------
